In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [ ]:
class PatchEmbed(nn.Module):
    """ Image to Patch Embedding
    """
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        img_size = (img_size,img_size)
        patch_size = (patch_size, patch_size)
        self.patch_shape = (img_size[0] // patch_size[0], img_size[1] // patch_size[1])
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        # The following variables are used in detection mycheckpointer.py
        self.num_patches = (img_size[1] // patch_size[1]) * (img_size[0] // patch_size[0])
        self.num_patches_w = self.patch_shape[0]
        self.num_patches_h = self.patch_shape[1]

    def forward(self, x, position_embedding=None):
        x = self.proj(x)   # 。shape should be (B, embed_dim, H, W)

        if position_embedding is not None:
            # interpolate the position embedding to the corresponding size
            position_embedding = position_embedding.view(1, self.patch_shape[0], self.patch_shape[1], -1).permute(0, 3, 1, 2)
            Hp, Wp = x.shape[2], x.shape[3]
            position_embedding = F.interpolate(position_embedding, size=(Hp, Wp), mode='bicubic')
            x = x + position_embedding

        x = x.flatten(2).transpose(1, 2) # (B, H*W, embed_dim)
        return x
    
    
patchembed = PatchEmbed()    
img = torch.randn(1, 3, 224, 224)  # Example input image
###--------- 我想给图像加上位置信息 ---------###


###

output = patchembed(img)
print(output.shape)

torch.Size([1, 196, 768])


#### **有关位置编码的几种常见方法

## 1. 固定位置方式嵌入 （sine position embedding）

In [8]:
import torch
import torch.nn as nn
import math

def get_2d_sincos_pos_embed(embed_dim, grid_size):
    """
    Generate 2D sine-cosine position embedding.
    Args:
        embed_dim (int): Embedding dimension.
        grid_size (tuple): (H, W) size of the grid.
    Returns:
        pos_embed (Tensor): (H*W, embed_dim) position embedding.
    """
    H, W = grid_size
    assert embed_dim % 2 == 0, "Embedding dimension must be even"
    
    # Create a grid of coordinates
    grid_h = torch.arange(H, dtype=torch.float32)
    grid_w = torch.arange(W, dtype=torch.float32)
    y, x = torch.meshgrid(grid_h, grid_w)
    
    # Normalize to [0, 1]
    y = y / (H - 1)
    x = x / (W - 1)
    
    # Generate sine and cosine embeddings
    pos_embed = torch.zeros(H, W, embed_dim)
    for i in range(embed_dim // 2):
        freq = 2 ** (i / (embed_dim // 2))
        pos_embed[:, :, 2 * i] = torch.sin(2 * math.pi * x * freq)
        pos_embed[:, :, 2 * i + 1] = torch.cos(2 * math.pi * y * freq)
    
    # Flatten the grid
    pos_embed = pos_embed.view(H * W, embed_dim)
    return pos_embed

# Example usage
embed_dim = 768
grid_size = (14, 14)  # For img_size=224 and patch_size=16
position_embedding = get_2d_sincos_pos_embed(embed_dim, grid_size)
print(position_embedding.shape)  # Should be (196, 768) for a 14x14 grid

torch.Size([196, 768])


## 2. 可学习位置方式嵌入 （learned position embedding）

In [9]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        self.patch_shape = (img_size[0] // patch_size[0], img_size[1] // patch_size[1])
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        
        # Initialize learnable position embedding
        self.position_embedding = nn.Parameter(torch.randn(1, embed_dim, self.patch_shape[0], self.patch_shape[1]))

    def forward(self, x):
        x = self.proj(x)  # (B, embed_dim, H, W)
        
        # Add position embedding
        x = x + self.position_embedding
        
        x = x.flatten(2).transpose(1, 2)  # (B, H*W, embed_dim)
        return x
    
embed_dim = 768
grid_size = (14, 14)
position_embedding = nn.Parameter(torch.randn(1, embed_dim, grid_size[0], grid_size[1]))
print(position_embedding.shape)

torch.Size([1, 768, 14, 14])


## 3. 相对位置方式嵌入 （Relative position embedding）

In [ ]:
class RelativePositionEmbedding(nn.Module):
    def __init__(self, embed_dim, patch_size):
        super().__init__()
        self.embed_dim = embed_dim
        self.patch_size = patch_size
        self.rel_pos = nn.Parameter(torch.randn(patch_size * 2 - 1, patch_size * 2 - 1, embed_dim))

    def forward(self, x):
        # Extract relative position embeddings
        B, N, C = x.shape
        H = W = int(math.sqrt(N))
        x = x.view(B, H, W, C)
        
        # Compute relative positions
        rel_pos = self.rel_pos
        rel_pos = rel_pos.view(1, self.patch_size * 2 - 1, self.patch_size * 2 - 1, C)
        rel_pos = rel_pos.expand(B, -1, -1, -1)
        
        # Apply relative position embeddings
        x = x + rel_pos
        x = x.view(B, N, C)
        return x